<a href="https://www.kaggle.com/code/anishamehajabin/dna-sequence-detective-mini-blas?scriptVersionId=369095826" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# DNA Sequence Detective — Mini-BLAST
A simplified BLAST seed-and-extend algorithm (ungapped) that finds which database sequence best matches an unknown DNA query.

**Pipeline:** k-mers → lookup table → seeds → ungapped X-drop extension → HSPs → ranking.

### Cell 1 — Imports and scoring constants
Match = +2, mismatch = −1, X-drop = 3 (gaps are not allowed). `defaultdict(list)` lets the lookup table store many positions per k-mer.

In [1]:
from collections import defaultdict

MATCH = 2
MISMATCH = -1
X_DROP = 3

### Cell 2 — Step 1: Generate k-mers
Slides a window of length `k` over the sequence and returns every overlapping word with its start position (there are `len(seq) - k + 1` of them).

In [2]:
def generate_kmers(seq, k):
    """Step 1: every overlapping substring of length k, with its start position."""
    return [(i, seq[i:i + k]) for i in range(len(seq) - k + 1)]

### Cell 3 — Step 2: Build the lookup table
Maps each distinct query k-mer to **all** of its start positions, so repeated words are kept, not discarded.

In [3]:
def build_lookup_table(query, k):
    """Step 2: k-mer -> list of ALL its start positions in the query."""
    table = defaultdict(list)
    for pos, word in generate_kmers(query, k):
        table[word].append(pos)
    return table

### Cell 4 — Step 3: Find exact seeds
Scans every k-mer of the database sequence and, if it exists in the lookup table, records each `(query_pos, subject_pos)` pair as a seed.

In [4]:
def find_seeds(table, subject, k):
    """Step 3: every (query_pos, subject_pos) pair sharing an identical k-mer."""
    seeds = []
    for s_pos, word in generate_kmers(subject, k):
        for q_pos in table.get(word, []):
            seeds.append((q_pos, s_pos))
    return seeds

### Cell 5 — Ungapped extension in one direction (X-drop)
Walks along the same diagonal one base pair at a time (`step=+1` right, `-1` left). The running score gets +2/−1; the best score and its length update only on a **strict** improvement. It stops when `best − current ≥ 3` or a sequence boundary is hit, and returns the length at the best score — not the last scanned position.

In [5]:
def extend_one_direction(query, subject, q_start, s_start, step):
    """Walk along the diagonal (step=+1 right, -1 left) using the X-drop rule.
    Returns (best additional score, extension length at that best score)."""
    current = best = 0
    best_len = length = 0
    qi, si = q_start, s_start
    while 0 <= qi < len(query) and 0 <= si < len(subject):
        current += MATCH if query[qi] == subject[si] else MISMATCH
        length += 1
        if current > best:            # update only on strict improvement
            best, best_len = current, length
        if best - current >= X_DROP:  # X-drop: stop this direction
            break
        qi += step
        si += step
    return best, best_len

### Cell 6 — Step 4: Extend a full seed
Extends left (starting just before the seed) and right (just after it) independently. Final score = `2k + best_left + best_right`. Coordinates are zero-based with exclusive ends.

In [6]:
def extend_seed(query, subject, q_pos, s_pos, k):
    """Step 4: ungapped extension of one seed. Returns (score, q_start, q_end, s_start, s_end)
    with zero-based, end-exclusive coordinates."""
    left_score, left_len = extend_one_direction(query, subject, q_pos - 1, s_pos - 1, -1)
    right_score, right_len = extend_one_direction(query, subject, q_pos + k, s_pos + k, +1)
    score = 2 * k + left_score + right_score
    return (score, q_pos - left_len, q_pos + k + right_len,
            s_pos - left_len, s_pos + k + right_len)

### Cell 7 — Step 5: Identify HSPs
Extends every seed, keeps only alignments with score ≥ H, and stores them in a dictionary keyed by the 4 coordinates `(q_start, q_end, s_start, s_end)` — so overlapping seeds giving the same alignment count as **one** HSP.

In [7]:
def find_hsps(query, subject, table, k, H):
    """Step 5: extend all seeds, keep score >= H, dedupe by the 4 coordinates."""
    hsps = {}
    for q_pos, s_pos in find_seeds(table, subject, k):
        score, qs, qe, ss, se = extend_seed(query, subject, q_pos, s_pos, k)
        if score >= H:
            hsps[(qs, qe, ss, se)] = score
    return hsps

### Cell 8 — Step 6: Rank database sequences
Each sequence is evaluated independently. The winner has the highest HSP score; using strict `>` means a tie keeps the earlier (smaller) index. Returns `-1, 0, 0` if nothing qualifies.

In [8]:
def solve_case(query, database, k, H):
    """Step 6: rank database sequences by best HSP score (ties -> smaller index)."""
    table = build_lookup_table(query, k)
    best_idx, best_score, best_count = -1, 0, 0
    for idx, subject in enumerate(database, start=1):
        hsps = find_hsps(query, subject, table, k, H)
        if not hsps:
            continue
        top = max(hsps.values())
        if top > best_score:          # strict '>' keeps the smaller index on ties
            best_idx, best_score, best_count = idx, top, len(hsps)
    return best_idx, best_score, best_count

### Cell 9 — Input parsing and output formatting
Reads T test cases (N, k, H, query, N subjects) from text and prints the result in the required `Case X:` format.

In [9]:
def run(text):
    tokens = text.split()
    pos = 0
    T = int(tokens[pos]); pos += 1
    out = []
    for case in range(1, T + 1):
        N, k, H = int(tokens[pos]), int(tokens[pos + 1]), int(tokens[pos + 2]); pos += 3
        query = tokens[pos]; pos += 1
        database = tokens[pos:pos + N]; pos += N
        y, s, c = solve_case(query, database, k, H)
        out.append(f"Case {case}:\nBest Match: {y}\nScore: {s}\nHSP Count: {c}")
    return "\n".join(out)

### Cell 10 — Run on the sample input
Expected: Case 1 → Best Match 1, Score 16, HSP Count 1; Case 2 → −1, 0, 0.

In [10]:
sample_input = """2
3 3 8
ATGCTAGC
TTATGCTAGCAA
GGGCTAGTT
CCCCAAAAG
2 3 10
ATGCTAGC
TTTTTTTT
GGGGGGGG
"""
print(run(sample_input))

Case 1:
Best Match: 1
Score: 16
HSP Count: 1
Case 2:
Best Match: -1
Score: 0
HSP Count: 0


### Cell 11 — Inspect the HSPs found (table view)
Shows two tables for Case 1: the query lookup table (each k-mer and its positions) and every distinct HSP per subject with its coordinates (0-based, end-exclusive) and score. Subject 2's best HSP scores 10 and Subject 1's scores 16, so Subject 1 wins.

In [11]:
import pandas as pd

query, k, H = "ATGCTAGC", 3, 8
subjects = ["TTATGCTAGCAA", "GGGCTAGTT", "CCCCAAAAG"]
table = build_lookup_table(query, k)

# Table 1: lookup table (k-mer -> all query positions)
lookup_df = pd.DataFrame([(w, pos) for w, pos in table.items()],
                         columns=["k-mer", "Query positions"])
print("Lookup table")
display(lookup_df.style.hide(axis="index"))

# Table 2: distinct HSPs per subject
rows = []
for i, subj in enumerate(subjects, start=1):
    hsps = find_hsps(query, subj, table, k, H)
    if not hsps:
        rows.append({"Subject": i, "Sequence": subj, "Query start": "-", "Query end": "-",
                     "Subject start": "-", "Subject end": "-", "Score": "No HSP"})
    for (qs, qe, ss, se), score in hsps.items():
        rows.append({"Subject": i, "Sequence": subj, "Query start": qs, "Query end": qe,
                     "Subject start": ss, "Subject end": se, "Score": score})
print("HSPs found (score >= H)")
display(pd.DataFrame(rows).style.hide(axis="index"))

Lookup table


k-mer,Query positions
ATG,[0]
TGC,[1]
GCT,[2]
CTA,[3]
TAG,[4]
AGC,[5]


HSPs found (score >= H)


Subject,Sequence,Query start,Query end,Subject start,Subject end,Score
1,TTATGCTAGCAA,0,8,2,10,16
2,GGGCTAGTT,2,7,2,7,10
3,CCCCAAAAG,-,-,-,-,No HSP


### Cell 12 — Try your own input
Edit the text below to test any sequences (useful for the code demonstration).

In [12]:
my_input = """1
2 4 10
ACGTACGTAA
TTACGTACGTAATT
ACGTTTTT
"""
print(run(my_input))

Case 1:
Best Match: 1
Score: 20
HSP Count: 3
